In [2]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob

from limb_fitting import *
from ellipse import *
from utils import *

In [3]:
def stretch(x, y, amount=1, angle=0, center=(0,0)):
    xc, yc = center
    r = amount
    a = angle
    R = np.array([[1 + np.cos(2 * a), np.sin(2 * a), -((1 + np.cos(2 * a)) * xc + np.sin(2 * a) * yc)],
                  [np.sin(2 * a), 1 - np.cos(2 * a), -(np.sin(2 * a) * xc + (1 - np.cos(2 * a)) * yc)],
                  [0, 0, 0]])
    R = np.identity(3) + (r - 1) / 2 * R
    xn = R[0,0] * x + R[0,1] * y + R[0,2]
    yn = R[1,0] * x + R[1,1] * y + R[1,2]
    return xn, yn

def undistort(x, y, coefficient=0, center=(0,0)):
    xc, yc = center
    k = coefficient

    r2 = (x - xc) ** 2 + (y - yc) ** 2
    q = 1 / (1 + k * r2)
    return xc + (x - xc) * q, yc + (y - yc) * q

def distort(x, y, coefficient=0, center=(0,0)):
    xc, yc = center
    k = coefficient

    r2 = (x - xc) ** 2 + (y - yc) ** 2
    q = (1 - np.sqrt(1 - 4 * k * r2)) / (2 * k * r2 + 1e-16)
    return xc + (x - xc) * q, yc + (y - yc) * q

In [4]:
files = sorted(glob.glob('/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/*.fits.gz'))
files

['/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/solo_L1_phi-fdt-ilam_20250923T000503_V202512011636C_0569230100.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/solo_L1_phi-fdt-ilam_20250923T001105_V202511212307C_0569230125.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/solo_L1_phi-fdt-ilam_20250923T001705_V202511212307C_0569230150.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/solo_L1_phi-fdt-ilam_20250923T002305_V202511212307C_0569230175.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/solo_L1_phi-fdt-ilam_20250923T002905_V202511212307C_0569230200.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/solo_L1_phi-fdt-ilam_20250923T003505_V202511212307C_0569230225.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/solo_L1_phi-fdt-ilam_20250923T004105_V202511212338C_0569230250.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025

In [5]:
xi, yi = np.mgrid[:2048,:2048]

xd, yd = distort(xi, yi, -2e-8, (937, 1055))
xd, yd = stretch(xd, yd, 1.001, -0.567, (1023.5, 1023.5)) ### Distorted grid

dxd = xd - xi, yd - yi

In [5]:
for file in files[:]:

    with fits.open(file) as hdul:
        image  = hdul[0].data[0]
        header = hdul[0].header

    image = bilinear(image, xd, yd)

    edges = find_edges(image)
    x, y = np.where(edges)
    x, y = filter_outliers(x, y, acc=1)
    ellipse = fit_ellipse(x, y)

    print(ellipse.radius, ellipse.flatness)

814.3747960059936 4.3281466969746774e-05
814.057461685993 2.20849920574695e-05
814.0937311716491 2.017914533858356e-05
814.1563104095135 3.103805043247476e-05
814.1801268774549 0.00032087897130828047
813.6782500592421 6.31072216273898e-05
813.7053460745212 1.2778776914523782e-05
813.7512208655342 2.081033784495645e-05
813.7881923201356 0.00024368016273212945


In [21]:
xi, yi = np.mgrid[:2048,:2048]

xu, yu = stretch(xi, yi, 1 / 1.001, -0.567, (1023.5, 1023.5))
xu, yu = undistort(xu, yu, -2e-8, (937, 1055)) ### Undistorted grid

dxu, dyu = xu - xi, yu - yi

In [7]:
for file in files[:]:

    with fits.open(file) as hdul:
        image  = hdul[0].data[0]
        header = hdul[0].header

    edges = find_edges(image)
    x, y = xu[edges], yu[edges]
    x, y = filter_outliers(x, y, acc=1)
    ellipse = fit_ellipse(x, y)

    print(ellipse.radius, ellipse.flatness)

814.3764366416442 4.3157302394347674e-05
814.0708670145841 4.299643691352273e-05
814.0983452571223 3.832034621487068e-05
814.1541826324628 2.730913982862404e-05
814.1827765961771 0.0003099586172597979
813.6868122678159 5.3577061236764756e-05
813.7186951787355 2.5798863482418e-05
813.7551278876397 3.080447404857356e-05
813.7980484323947 0.0002592011386217674


In [8]:
#np.savez('distortion.npz', xd=xd, yd=yd, xu=xu, yu=yu)

In [14]:
files = sorted(glob.glob('/home/ulyanov/data/cross/*'))
file = files[1]

In [53]:
with fits.open(file) as hdul:
    image  = hdul[0].data[0]
    header = hdul[0].header

nx, ny = header['NAXIS2'], header['NAXIS1']
x0, y0 = header['PXBEG2'] - 1, header['PXBEG1'] - 1
xd_, yd_ = xd[x0:x0+nx, y0:y0+ny] - x0, yd[x0:x0+nx, y0:y0+ny] - y0
xu_, yu_ = xu[x0:x0+nx, y0:y0+ny] - x0, yu[x0:x0+nx, y0:y0+ny] - y0

edges = find_edges(image)
#x, y = np.where(edges)
x, y = xu_[edges], yu_[edges]
x, y = filter_outliers(x, y, acc=1)
ellipse = fit_ellipse(x, y)

ellipse.center, ellipse.radius, ellipse.flatness

((np.float64(421.6412747518126), np.float64(457.8734726477365)),
 np.float64(330.257008863921),
 np.float64(0.00024538804054119456))

In [39]:
plt.figure(figsize=(10,10))
plt.plot(x,y, '.')

In [19]:
find_center(image)

(np.float64(421.644809641154),
 np.float64(457.8726375258009),
 np.float64(330.2539645678348))